In [1]:
import torch
import transformers

print("Environment working")

Environment working


In [2]:
from datasets import load_dataset

dataset = load_dataset("squad_v2")

print(dataset)

README.md: 0.00B [00:00, ?B/s]

squad_v2/train-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

squad_v2/validation-00000-of-00001.parqu(…):   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 130319
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11873
    })
})


In [3]:
dataset["validation"]

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 11873
})

In [9]:
example = dataset["validation"][0]

print(example)

{'id': '56ddde6b9a695914005b9628', 'title': 'Normans', 'context': 'The Normans (Norman: Nourmands; French: Normands; Latin: Normanni) were the people who in the 10th and 11th centuries gave their name to Normandy, a region in France. They were descended from Norse ("Norman" comes from "Norseman") raiders and pirates from Denmark, Iceland and Norway who, under their leader Rollo, agreed to swear fealty to King Charles III of West Francia. Through generations of assimilation and mixing with the native Frankish and Roman-Gaulish populations, their descendants would gradually merge with the Carolingian-based cultures of West Francia. The distinct cultural and ethnic identity of the Normans emerged initially in the first half of the 10th century, and it continued to evolve over the succeeding centuries.', 'question': 'In what country is Normandy located?', 'answers': {'text': ['France', 'France', 'France', 'France'], 'answer_start': [159, 159, 159, 159]}}


In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

In [11]:
model_name = "deepset/bert-base-cased-squad2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: deepset/bert-base-cased-squad2
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForQuestionAnswering(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elem

In [12]:
context = example["context"]
question = example["question"]

inputs = tokenizer(question, context, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

start_index = torch.argmax(outputs.start_logits)
end_index = torch.argmax(outputs.end_logits)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
predicted_answer = tokenizer.convert_tokens_to_string(tokens[start_index:end_index+1])

true_answer = example["answers"]["text"][0]

print("Question:", question)
print("Predicted Answer:", predicted_answer)
print("True Answer:", true_answer)

Question: In what country is Normandy located?
Predicted Answer: France
True Answer: France


In [35]:
results = []

for i in range(len(df)):
    example = dataset["validation"][i]

    context = example["context"]
    question = example["question"]
    true_answers = example["answers"]["text"]

    inputs = tokenizer(question, context, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    start_logits = outputs.start_logits
    end_logits = outputs.end_logits

    start_index = torch.argmax(start_logits)
    end_index = torch.argmax(end_logits)

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    predicted_answer = tokenizer.convert_tokens_to_string(tokens[start_index:end_index+1])

    true_answer = true_answers[0] if len(true_answers) > 0 else ""

    correct = predicted_answer.lower() == true_answer.lower()

    results.append({
        "question": question,
        "predicted": predicted_answer,
        "true": true_answer,
        "correct": correct,
        "start_logits": start_logits.numpy(),
        "end_logits": end_logits.numpy()
    })

df = pd.DataFrame(results)

df.head()

,question,predicted,true,correct,start_logits,end_logits
0,In what country is Normandy located?,France,France,True,"[[-3.02749, -8.174801, -8.407998, -8.194277, -...","[[-2.380471, -8.395869, -7.7617874, -7.942414,..."
1,When were the Normans in Normandy?,10th and 11th centuries,10th and 11th centuries,True,"[[0.4090171, -7.866979, -8.080148, -8.300548, ...","[[-0.34856105, -7.8766556, -8.135746, -8.43703..."
2,From which countries did the Norse originate?,,"Denmark, Iceland and Norway",False,"[[6.604309, -8.025348, -8.379556, -8.980244, -...","[[6.2046175, -8.419378, -7.6635, -7.2663493, -..."
3,Who was the Norse leader?,Rollo,Rollo,True,"[[6.415034, -7.598161, -8.306515, -7.793644, -...","[[4.9600334, -7.9587026, -8.153611, -8.915671,..."
4,What century did the Normans first gain their ...,10th,10th century,False,"[[-0.23108938, -7.7582216, -8.784559, -8.17281...","[[0.46058223, -8.097748, -7.632112, -8.205743,..."


In [36]:
import torch
import torch.nn.functional as F

span_probs = []

for i in range(len(df)):
    
    start_logits = torch.tensor(df.loc[i, "start_logits"]).squeeze()
    end_logits = torch.tensor(df.loc[i, "end_logits"]).squeeze()

    start_probs = F.softmax(start_logits, dim=-1)
    end_probs = F.softmax(end_logits, dim=-1)

    start_index = torch.argmax(start_logits).item()
    end_index = torch.argmax(end_logits).item()

    span_prob = start_probs[start_index] * end_probs[end_index]

    span_probs.append(span_prob.item())

print("Span probabilities calculated")

df["span_prob"] = span_probs

df.head()

Span probabilities calculated


,question,predicted,true,correct,start_logits,end_logits,span_prob
0,In what country is Normandy located?,France,France,True,"[[-3.02749, -8.174801, -8.407998, -8.194277, -...","[[-2.380471, -8.395869, -7.7617874, -7.942414,...",0.999525
1,When were the Normans in Normandy?,10th and 11th centuries,10th and 11th centuries,True,"[[0.4090171, -7.866979, -8.080148, -8.300548, ...","[[-0.34856105, -7.8766556, -8.135746, -8.43703...",0.716860
2,From which countries did the Norse originate?,,"Denmark, Iceland and Norway",False,"[[6.604309, -8.025348, -8.379556, -8.980244, -...","[[6.2046175, -8.419378, -7.6635, -7.2663493, -...",0.416920
3,Who was the Norse leader?,Rollo,Rollo,True,"[[6.415034, -7.598161, -8.306515, -7.793644, -...","[[4.9600334, -7.9587026, -8.153611, -8.915671,...",0.501952
4,What century did the Normans first gain their ...,10th,10th century,False,"[[-0.23108938, -7.7582216, -8.784559, -8.17281...","[[0.46058223, -8.097748, -7.632112, -8.205743,...",0.361124


In [37]:
import numpy as np

start_entropies = []
end_entropies = []

for i in range(len(df)):
    
    start_logits = torch.tensor(df.loc[i, "start_logits"]).squeeze()
    end_logits = torch.tensor(df.loc[i, "end_logits"]).squeeze()
    
    start_probs = torch.softmax(start_logits, dim=-1).numpy()
    end_probs = torch.softmax(end_logits, dim=-1).numpy()
    
    start_entropy = -np.sum(start_probs * np.log(start_probs + 1e-12))
    end_entropy = -np.sum(end_probs * np.log(end_probs + 1e-12))
    
    start_entropies.append(start_entropy)
    end_entropies.append(end_entropy)

df["start_entropy"] = start_entropies
df["end_entropy"] = end_entropies

df.head()

,question,predicted,true,correct,start_logits,end_logits,span_prob,start_entropy,end_entropy
0,In what country is Normandy located?,France,France,True,"[[-3.02749, -8.174801, -8.407998, -8.194277, -...","[[-2.380471, -8.395869, -7.7617874, -7.942414,...",0.999525,0.001243,0.003745
1,When were the Normans in Normandy?,10th and 11th centuries,10th and 11th centuries,True,"[[0.4090171, -7.866979, -8.080148, -8.300548, ...","[[-0.34856105, -7.8766556, -8.135746, -8.43703...",0.716860,0.790975,0.035410
2,From which countries did the Norse originate?,,"Denmark, Iceland and Norway",False,"[[6.604309, -8.025348, -8.379556, -8.980244, -...","[[6.2046175, -8.419378, -7.6635, -7.2663493, -...",0.416920,0.694885,0.575946
3,Who was the Norse leader?,Rollo,Rollo,True,"[[6.415034, -7.598161, -8.306515, -7.793644, -...","[[4.9600334, -7.9587026, -8.153611, -8.915671,...",0.501952,0.690123,0.468602
4,What century did the Normans first gain their ...,10th,10th century,False,"[[-0.23108938, -7.7582216, -8.784559, -8.17281...","[[0.46058223, -8.097748, -7.632112, -8.205743,...",0.361124,1.170431,0.978716


In [38]:
margins = []

for i in range(len(df)):
    
    start_logits = torch.tensor(df.loc[i, "start_logits"]).squeeze()
    start_probs = torch.softmax(start_logits, dim=-1)
    
    top2 = torch.topk(start_probs, 2).values
    
    margin = (top2[0] - top2[1]).item()
    
    margins.append(margin)

df["confidence_margin"] = margins

df.head()

,question,predicted,true,correct,start_logits,end_logits,span_prob,start_entropy,end_entropy,confidence_margin
0,In what country is Normandy located?,France,France,True,"[[-3.02749, -8.174801, -8.407998, -8.194277, -...","[[-2.380471, -8.395869, -7.7617874, -7.942414,...",0.999525,0.001243,0.003745,0.999882
1,When were the Normans in Normandy?,10th and 11th centuries,10th and 11th centuries,True,"[[0.4090171, -7.866979, -8.080148, -8.300548, ...","[[-0.34856105, -7.8766556, -8.135746, -8.43703...",0.716860,0.790975,0.035410,0.524292
2,From which countries did the Norse originate?,,"Denmark, Iceland and Norway",False,"[[6.604309, -8.025348, -8.379556, -8.980244, -...","[[6.2046175, -8.419378, -7.6635, -7.2663493, -...",0.416920,0.694885,0.575946,0.030302
3,Who was the Norse leader?,Rollo,Rollo,True,"[[6.415034, -7.598161, -8.306515, -7.793644, -...","[[4.9600334, -7.9587026, -8.153611, -8.915671,...",0.501952,0.690123,0.468602,0.202216
4,What century did the Normans first gain their ...,10th,10th century,False,"[[-0.23108938, -7.7582216, -8.784559, -8.17281...","[[0.46058223, -8.097748, -7.632112, -8.205743,...",0.361124,1.170431,0.978716,0.594383


In [39]:
df.groupby("correct")[["span_prob","start_entropy","end_entropy","confidence_margin"]].mean()

,span_prob,start_entropy,end_entropy,confidence_margin
correct,,,,
False,0.696004,0.581662,0.536497,0.705813
True,0.707652,0.532147,0.449297,0.625170


In [ ]:
df["entropy_mean"] = (df["start_entropy"] + df["end_entropy"]) / 2
df["entropy_diff"] = abs(df["start_entropy"] - df["end_entropy"])

df.head()

In [ ]:
df.to_csv("../results/qa_uncertainty_signals.csv", index=False)
print("CSV updated")

In [ ]:
features = [
    "span_prob",
    "start_entropy",
    "end_entropy",
    "confidence_margin",
    "entropy_mean",
    "entropy_diff"
]

X = df[features]
y = df["correct"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train, y_train)

print("Random Forest trained")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

In [ ]:
import pandas as pd

importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf_model.feature_importances_
})

importance = importance.sort_values(by="importance", ascending=False)

importance

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.barplot(
    x=importance["importance"],
    y=importance["feature"]
)

plt.title("Feature Importance for Reliability Prediction")
plt.show()

In [ ]:
import joblib

joblib.dump(rf_model, "../results/reliability_classifier.pkl")

print("Model saved successfully")

In [ ]:
import joblib

rf_model = joblib.load("../results/reliability_classifier.pkl")

print("Model loaded successfully")

In [ ]:
joblib.dump(features, "../results/feature_order.pkl")